In [0]:
from pyspark.sql import SparkSession

# Spark Config
spark = SparkSession.builder \
    .appName("PySparkETL") \
    .master("local[*]") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .config("spark.driver.memory", "2g") \
    .config("spark.driver.cores", "1") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

In [0]:
bookings_df = spark.read.csv(path="dbfs:/FileStore/bookings.csv",
                    sep=',',
                    header=True, inferSchema=True)
members_df = spark.read.csv(path="dbfs:/FileStore/members.csv",
                    sep=',',
                    header=True, inferSchema=True)
facilities_df = spark.read.csv(path="dbfs:/FileStore/facilities.csv",
                    sep=',',
                    header=True, inferSchema=True)

In [0]:
# ETL bookings.csv file
# Extract: Load data from CSV file into a DF
df = spark.read.csv(path="dbfs:/FileStore/bookings.csv",
                    sep=',',
                    header=True, inferSchema=True)

    
# Drop the table if it already exists
spark.sql("DROP TABLE IF EXISTS bookings")

# Load: Write data from DataFrame into managed table
df.write.saveAsTable("bookings")

In [0]:
# ETL facilities.csv file
# Extract: Load data from CSV file into a DF
df = spark.read.csv(path="dbfs:/FileStore/facilities.csv",
                    sep=',',
                    header=True, inferSchema=True)

    
# Drop the table if it already exists
spark.sql("DROP TABLE IF EXISTS facilities")

# Load: Write data from DataFrame into managed table
df.write.saveAsTable("facilities")

In [0]:
# ETL members.csv file
# Extract: Load data from CSV file into a DF
df = spark.read.csv(path="dbfs:/FileStore/members.csv",
                    sep=',',
                    header=True, inferSchema=True)

    
# Drop the table if it already exists
spark.sql("DROP TABLE IF EXISTS members")

# Load: Write data from DataFrame into managed table
df.write.saveAsTable("members")

## ETL Job One: Parquet file

In [0]:
# Extract data from tables
bookings_df = spark.table("bookings_csv")
members_df = spark.table("members_csv")
facilities_df = spark.table("facilities_csv")

# Transform data
# Produce a list of the total number of slots booked per facility in the month of September 2012
filtered_bookings = bookings_df.filter(bookings_df.starttime.contains('2012-09'))

slots_per_facility = filtered_bookings.groupBy("facid").agg(F.sum("slots").alias("slots"))
results_df = slots_per_facility.orderBy("slots", ascending=True)

# Load the result into a Parquet file
output_path = "dbfs:/FileStore/output/results_df.parquet"
results_df.write.parquet(output_path)

## ETL Job Two: Partitions

In [0]:
# Extract data from managed tables
bookings_df = spark.table("bookings_csv")
members_df = spark.table("members_csv")
facilities_df = spark.table("facilities_csv")

# Transform data
# Produce a list of all members who have used a tennis court
joined1_df = members_df.join(bookings_df, members_df.memid == bookings_df.memid, 'left_outer')

joined2_df = joined1_df.join(facilities_df, facilities_df.facid == joined1_df.facid, 'left_outer').filter(facilities_df.name.contains("Tennis Court"))

results_df = joined2_df.select(members_df.firstname, members_df.surname, facilities_df.name).distinct().orderBy(members_df.firstname)

# Load into a Delta table partitioned
results_df.write.format("delta").partitionBy(facilities_df.name).mode("overwrite").saveAsTable("threejoin_delta")

## ETL Job Three: HTTP Requests

In [0]:
# Extract daily stock price data price from Google, Apple, Microsoft, and Tesla
# Function to fetch data from API
def fetch_stock_data(symbol, api_key):
    url = "https://alpha-vantage.p.rapidapi.com/query"
    querystring = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "datatype": "json",
        "outputsize": "compact"
    }
    headers = {
        "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
        "X-RapidAPI-Key": api_key
    }
    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()
    return data

companies = {
    "Google": "GOOGL",
    "Apple": "AAPL",
    "Microsoft": "MSFT",
    "Tesla": "TSLA"
}
api_key = "54706e6343msh96825dc87a0b044p188036jsn4bab408b03a2"

dfs = []
for company, symbol in companies.items():
    data = fetch_stock_data(symbol, api_key)
    daily_series = data['Time Series (Daily)']
    
    # Convert to pandas DataFrame first
    df = pd.DataFrame(daily_series).T.reset_index()
    df.columns = ['date', 'open', 'high', 'low', 'close', 'adjusted_close', 'volume', 'dividend_amount', 'split_coefficient']
    df['company'] = company
    
    # Convert pandas DataFrame to Spark DataFrame
    spark_df = spark.createDataFrame(df)
    dfs.append(spark_df)

# Union all DataFrames
all_data_df = dfs[0]
for df in dfs[1:]:
    all_data_df = all_data_df.union(df)

# Transform data
all_data_df = all_data_df.withColumn("date", to_date(col("date"))) \
                         .withColumn("year", year(col("date"))) \
                         .withColumn("week", weekofyear(col("date")))

weekly_max_df = all_data_df.groupBy("company", "year", "week") \
                           .agg(spark_max(col("close")).alias("max_closing_price")) \
                           .orderBy("company", "year", "week")

# Load data
weekly_max_df.write.format("delta").partitionBy("company").mode("overwrite").saveAsTable("max_closing_price_weekly")

## ETL Job Four: RDBMS

In [0]:
# Extract data from PostgreSQL database
# Database connection properties
url = "jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs"
properties = {
    "user": "reader",
    "password": "NWDMCE5xdipIjRrp",
    "driver": "org.postgresql.Driver"
}

query = "(SELECT * FROM rna LIMIT 100) AS rna_subset"
rna_df = spark.read.jdbc(url=url, table=query, properties=properties)
rna_df.show()

# Load the DataFrame into a managed table
rna_df.write.format("delta").mode("overwrite").saveAsTable("rna_100_records")